# Fold illusions with factorized blending

The same idea as [`illusions.ipynb`](illusions.ipynb): one picture when the sheet lies flat, and another on each wing of the folded crane. Only the flat sheet and the finished crane are used, with no intermediate stages. The pictures here are all from Japan: Mt Fuji at sunset under sakura when flat, a koi carp on one wing, and a torii gate on the other.

**What's being tested is how visible pixels are shared.** In the original, a pixel that shows on the folded crane follows two prompts at once, and the sampler averages their predictions. Following [Factorized Diffusion](https://dangeng.github.io/factorized_diffusion/) (Geng, Park, Owens, ECCV 2024), this notebook can split the shared pixels by frequency instead:

- **The flat prompt** controls the sheet's low frequencies: its broad shapes and colors.
- **The wing prompts** control the high frequencies, wherever their paper shows.

A wing shows only a small patch of the sheet, so its whole picture is fine detail at the sheet's scale. That's why this split might suit folding. `factorized.py` runs `illusion.generate` unchanged and swaps only its blending step, so the original and the factorized blend can be compared on the same seed. The original notebook and its code are untouched.

At the end, CLIP scores every result: how well each view shows its own subject, and how well the flat sheet hides the wings' subjects.

**Status:** not yet run on a GPU with these prompts. The blend's math, the scoring, and a full run of the sampler with a stand-in model are checked by `tests/test_factorized.py`.

## Setup

In [ ]:
import importlib, os, subprocess, sys

FOLDER = "fold-map"
if not os.path.exists("factorized.py"):
    if not os.path.exists("folded-illusions"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/rath-hy/folded-illusions.git"], check=True)
    os.chdir(os.path.join("folded-illusions", FOLDER))
if "google.colab" in sys.modules:
    # On Colab, rerunning this cell gets the latest code and reloads anything already imported.
    subprocess.run(["git", "pull", "-q"], check=True)
    for name in ["fold_map", "decal", "illusion", "factorized"]:
        if name in sys.modules:
            importlib.reload(sys.modules[name])
sys.path.insert(0, os.getcwd())

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from fold_map import FoldedModel
from decal import test_picture
from illusion import FoldViews, canvas_picture, plot_result, printable, square_picture
import factorized

## The crane and its views

As in the original: the flat sheet, plus the folded crane seen from each side. The code calls those views front and back, and each shows one wing. Stage 1 works at 64 × 64, stage 2 at 256 × 256.

In [ ]:
model = FoldedModel("004_traditional_Crane_cp.fold", "004_traditional_Crane_state.fold")
views64, views256 = FoldViews(model, 64), FoldViews(model, 256)
for name, v in (("64", views64), ("256", views256)):
    print(name, v.stats())

## What the split does, without AI

Before spending GPU time, here's the same split applied to finished pictures. The letters are stand-ins: A for the flat picture, and F and B for the two wings.

`SIGMA` is the blur's width as a fraction of the sheet's side, and it's where the split happens. A small blur gives the flat picture almost everything, leaving the folded pictures only their sharpest edges. A larger one leaves the flat picture only the broad shapes, and gives the folded pictures more of their own color and shading. For scale, the crane's whole folded picture spans about 16 × 22 pixels at stage 1, about a quarter of the sheet's side.

Read each row left to right: the flat sheet, then each folded view.

In [ ]:
flat_pic = square_picture(test_picture("A", (60, 140, 90), (1000, 1000)), 256)
front_pic = canvas_picture(views256, test_picture("F", (196, 120, 60), (1300, 1000)), "front")
back_pic = canvas_picture(views256, test_picture("B", (52, 98, 168), (1300, 1000)), "back")

from illusion import composite
previews = {"composite (no blending)": composite(views256, flat_pic, front_pic, back_pic)}
for sigma in (1 / 64, 1 / 32, 1 / 16):
    previews[f"split at SIGMA = 1/{round(1 / sigma)}"] = factorized.preview(views256, flat_pic, front_pic, back_pic, sigma)
factorized.plot_comparison(views256, previews)
plt.show()

In the composite, the flat sheet is covered in sharp shards of the folded pictures. With the split, the shards keep their edges but take the flat picture's colors and shading, so they're harder to spot on the flat sheet. The folded views pay for that: their broad colors come from the flat picture too.

That's the trade we're testing. The generator can do better than this preview, because it draws all three pictures together and can choose content where the trade doesn't show.

## Generate (needs a GPU)

Setup matches `illusions.ipynb`, except that the install cell also upgrades protobuf.

1. Use a **GPU runtime** (Runtime → Change runtime type; a T4 works).
2. On Hugging Face, accept the licenses for [DeepFloyd/IF-I-M-v1.0](https://huggingface.co/DeepFloyd/IF-I-M-v1.0) and [DeepFloyd/IF-II-M-v1.0](https://huggingface.co/DeepFloyd/IF-II-M-v1.0), and have an [access token](https://huggingface.co/settings/tokens) ready.

In [ ]:
# protobuf 7.36 or later: current transformers ships a tokenizer file older protobuf can't read.
!pip install -q -U diffusers transformers accelerate sentencepiece "protobuf>=7.36"
import sys
if "google.protobuf" in sys.modules:
    print("protobuf was already loaded before this upgrade: choose Runtime → Restart session, then run from the top.")
from huggingface_hub import notebook_login
notebook_login()

### Prompts

Designed to mix well, from the set: sakura trees, sunset, the yen, koi carp, torii gate, Mt Fuji.

- **One style for all three: ukiyo-e woodblock prints.** A shared style hides the seams where pictures meet. Flat colors and bold outlines also read at thumbnail size, which matters because each wing's picture is laid out in about 16 × 22 pixels, and they survive home printing better than the first trial's dark oil painting.
- **Flat: Mt Fuji at sunset, framed by sakura.** A wide landscape with busy blossoms absorbs the wings' shards, as the city did.
- **Wings: a koi and a torii gate.** Both are bold, simple and recognizable small. Each prompt asks for a close-up, so the subject fills the paper.
- **Surroundings in the flat picture's colors.** The koi swims among pink sakura petals, and the torii stands against an orange sunset sky. Where a wing's paper shows on the flat sheet, its colors already belong there, which makes the shards harder to spot. This replaces the old "on a plain light background", which made pale patches with hard straight edges on the flat sheet.
- **Left out:** the yen. Its lettering and fine detail can't survive 16 × 22 pixels. The sunset lives in the flat scene and in the torii's sky.

`CLIP_TEXTS` are what the scoring checks each view for, without the style words.

In [ ]:
STYLE = "a ukiyo-e woodblock print of "
FLAT_PROMPT = STYLE + "Mount Fuji at sunset, framed by blooming sakura trees"
FRONT_PROMPT = STYLE + "a close-up of a large orange and white koi carp swimming among floating pink sakura petals"
BACK_PROMPT = STYLE + "a close-up of a red torii gate against a glowing orange sunset sky"

VIEW_TITLES = ("flat: Mt Fuji", "koi wing (front view)", "torii wing (back view)")
CLIP_TEXTS = ("Mount Fuji at sunset with sakura trees", "a koi carp", "a torii gate")

# The original prompts, to compare with illusions.ipynb:
# STYLE = "an oil painting of "
# FLAT_PROMPT = STYLE + "a crowded city street at night"
# FRONT_PROMPT = STYLE + "a red fox, on a plain light background"
# BACK_PROMPT = STYLE + "a white owl, on a plain light background"
# VIEW_TITLES = ("flat: city", "fox wing (front view)", "owl wing (back view)")
# CLIP_TEXTS = ("a city street at night", "a red fox", "a white owl")

### Encode the prompts

Rerun this cell whenever you change the prompts.

In [ ]:
import gc, torch
from transformers import T5EncoderModel
from diffusers import IFPipeline, IFSuperResolutionPipeline

# The safety checker and watermarker aren't used by our sampler, and the safety checker
# doesn't load with current versions of transformers, so skip them.
skip = dict(safety_checker=None, feature_extractor=None, watermarker=None)

text_encoder = T5EncoderModel.from_pretrained("DeepFloyd/IF-I-M-v1.0", subfolder="text_encoder", variant="fp16",
                                              torch_dtype=torch.float16, device_map="cuda")
encoder = IFPipeline.from_pretrained("DeepFloyd/IF-I-M-v1.0", text_encoder=text_encoder, unet=None, **skip)
prompt_embeds, negative_embeds = encoder.encode_prompt([FLAT_PROMPT, FRONT_PROMPT, BACK_PROMPT])

del encoder, text_encoder
gc.collect(); torch.cuda.empty_cache()
print("embeddings:", tuple(prompt_embeds.shape))

### Load the image models

This only needs to run once per session.

In [ ]:
if "stage_1" not in globals():
    stage_1 = IFPipeline.from_pretrained("DeepFloyd/IF-I-M-v1.0", text_encoder=None, variant="fp16",
                                         torch_dtype=torch.float16, **skip).to("cuda")
    stage_2 = IFSuperResolutionPipeline.from_pretrained("DeepFloyd/IF-II-M-v1.0", text_encoder=None, variant="fp16",
                                                        torch_dtype=torch.float16, **skip).to("cuda")

### Compare the blends

One run with the original averaging, then one per `SIGMA` with the factorized blend, all with the same seed. Everything else is identical, so differences come from the blend alone. Each run takes a minute or two on a T4.

- `SIGMAS`: where the split happens, as a fraction of the sheet's side.
- `LOW`: which prompt gets the low frequencies. `"flat"` is the idea above, and `"folded"` swaps the roles, as a control.
- `FOLD_WEIGHT`: used by the original average, and by both for the variance channels DeepFloyd also predicts.

Stage 1 decides each picture's layout, so its results are shown too.

In [ ]:
from illusion import generate as generate_averaged

SIGMAS = (1 / 64, 1 / 32, 1 / 16)
LOW = "flat"
FOLD_WEIGHT = 0.5
GUIDANCE = (7.0, 4.0)
STEPS = (50, 40)
SEED = 0

embeds = (negative_embeds, prompt_embeds)
settings = dict(guidance=GUIDANCE, steps=STEPS, fold_weight=FOLD_WEIGHT, seed=SEED)
results = {f"seed {SEED}, averaged": generate_averaged(stage_1, stage_2, views64, views256, embeds, **settings)}
for sigma in SIGMAS:
    results[f"seed {SEED}, factorized 1/{round(1 / sigma)}"] = factorized.generate(
        stage_1, stage_2, views64, views256, embeds, sigma_frac=sigma, low=LOW, **settings)

factorized.plot_comparison(views64, {k: s1 for k, (s1, _) in results.items()}, VIEW_TITLES); plt.suptitle("stage 1 (64 × 64)", y=1.01)
factorized.plot_comparison(views256, {k: s2 for k, (_, s2) in results.items()}, VIEW_TITLES); plt.suptitle("stage 2 (256 × 256)", y=1.01)
plt.show()

**What to look for:**

- **Flat:** does Mt Fuji stay a clean landscape, with fewer crane-shaped shards breaking it up?
- **Folded:** do the koi and the torii read clearly on their wings?
- **Seams:** creases cut the folded views, and those edges count as high frequency.

Seeds matter as much as settings in these illusions, so compare a few before concluding anything.

### Try more seeds

The original blend and one factorized setting, side by side, for several seeds. These are added to `results`, so the scoring below ranks them too.

In [ ]:
SEEDS = (1, 2, 3)
SIGMA = 1 / 32

for seed in SEEDS:
    s = dict(settings, seed=seed)
    pair = {f"seed {seed}, averaged": generate_averaged(stage_1, stage_2, views64, views256, embeds, **s),
            f"seed {seed}, factorized 1/{round(1 / SIGMA)}": factorized.generate(
                stage_1, stage_2, views64, views256, embeds, sigma_frac=SIGMA, low=LOW, **s)}
    results.update(pair)
    factorized.plot_comparison(views256, {k: s2 for k, (_, s2) in pair.items()}, VIEW_TITLES)
    plt.show()

### Score every result with CLIP

[CLIP](https://huggingface.co/openai/clip-vit-large-patch14) measures how well an image matches a piece of text. For each result:

- **flat, front, back:** how well each view matches its own subject. Higher is better.
- **leak:** how much the flat sheet looks like the wings' subjects. Lower is better.
- **score:** the mean of the first three, minus leak.

CLIP doesn't judge beauty, and differences of a few hundredths are noise. Use it to narrow the field, then pick by eye. The first run downloads CLIP, about 1.7 GB.

In [ ]:
if "embed_images" not in globals():
    embed_images, embed_texts = factorized.clip_embedders()
ranking = factorized.score_views(views256, {k: s2 for k, (_, s2) in results.items()}, embed_images, embed_texts, CLIP_TEXTS)

print(f"{'result':32} {'flat':>6} {'front':>6} {'back':>6} {'leak':>6} {'score':>6}")
for r in ranking:
    print(f"{r['label']:32} {r['flat']:6.3f} {r['front']:6.3f} {r['back']:6.3f} {r['leak']:6.3f} {r['score']:6.3f}")

### Fold the best one

By default this takes CLIP's top result. Look at it first, and set `BEST` to another label if your eye disagrees. This folds the result virtually with the decal simulator, which works independently of the views used to make it, then saves a print file. Print it at **actual size**, and fold with the **printed side out**: corner A becomes the tail, C the head, and B and D the wing tips.

In [ ]:
BEST = ranking[0]["label"]     # or any key from results, e.g. "seed 0, averaged"

sheet = results[BEST][1]
plot_result(views256, sheet, BEST)
plt.show()

name = "japan_" + BEST.replace("seed ", "seed").replace(", ", "_").replace(" ", "").replace("/", "-") + ".png"
printable(sheet).save(name, dpi=(300, 300))
print("saved", name)
try:
    from google.colab import files
    files.download(name)
except ImportError:
    pass

## Notes

- **The split is linear.** Like Factorized Diffusion, it changes the model's predictions, not the noisy image, so every view still sees properly noisy input.
- **Both stages use the same physical split.** `SIGMA` is a fraction of the sheet, so it's 2 pixels at stage 1 and 8 at stage 2 for 1/32. `factorized.generate(..., factorize=(True, False))` limits it to stage 1.
- **Hidden paper.** Far from any visible paper, the result is the flat prediction alone, as in the original. Within about a blur width of the visible regions, a little of the folded views' detail spreads into hidden paper.
- **CLIP is a rough guide.** It sees each view at 224 × 224 and knows nothing about printing or folding, so the final call is yours.
- **Nothing stops leaking.** As Factorized Diffusion notes, one prompt's content can still show up in another's component.

## References

- D. Geng, I. Park, A. Owens. [Factorized Diffusion: Perceptual Illusions by Noise Decomposition](https://dangeng.github.io/factorized_diffusion/). *ECCV*, 2024.
- D. Geng, I. Park, A. Owens. [Visual Anagrams: Generating Multi-View Optical Illusions with Diffusion Models](https://dangeng.github.io/visual_anagrams/). *CVPR*, 2024.
- DeepFloyd. [IF](https://github.com/deep-floyd/IF): a pixel-space cascaded text-to-image diffusion model, 2023.